# SASV: LFCC calibrate-then-fuse (B1-v2 style)

Reuses LFCC score CSVs (no GPU re-score):

- `runs/ecapa_plus_lfcc_dev/scores_dev.csv`
- `runs/ecapa_plus_lfcc_eval/scores_eval.csv`

**Idea (SASV Baseline1-v2 family)**  
Raw cosine and CM scores live on different scales. Fit **logistic (Platt) calibrators on dev**, then fuse calibrated scores. Lock calibrators; apply **once** on eval.

| Method | Fusion |
|--------|--------|
| `raw_sum` | `s_asv + s_cm` (your notebook 03/04 baseline) |
| `platt_sum_sasv` | `P(target\|s_asv) + P(target\|s_cm)` |
| `logit_sum_sasv` | logits sum (same calibrators) |
| `platt_sum_sv_spf` | ASV Platt on target/nontarget; CM Platt on bona/spoof |
| `joint_proba` / `joint_logit` | logistic on `[s_asv, s_cm]` |

Needs `scikit-learn` (already in `app/server` venv).

In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "calibrate_fusion_lib.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019")
sys.path.insert(0, str(ROOT))

from experiment_lib import DEFAULT_SASV, RUNS_DIR
from weighted_fusion_lib import load_score_csv
from calibrate_fusion_lib import (
    evaluate_methods,
    fit_calibrators_dev,
    fuse_scores,
    pick_best_method,
    save_calibrators,
    save_run_summary,
)

DEV_CSV = RUNS_DIR / "ecapa_plus_lfcc_dev" / "scores_dev.csv"
EVAL_CSV = RUNS_DIR / "ecapa_plus_lfcc_eval" / "scores_eval.csv"
OUT_DIR = RUNS_DIR / "ecapa_plus_lfcc_calibrated"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("dev:", DEV_CSV.exists(), DEV_CSV)
print("eval:", EVAL_CSV.exists(), EVAL_CSV)

dev: True D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019\runs\ecapa_plus_lfcc_dev\scores_dev.csv
eval: True D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019\runs\ecapa_plus_lfcc_eval\scores_eval.csv


## Knobs

- Fit / pick method on **dev** only
- `RUN_EVAL = True` applies the **locked** method once on eval

In [2]:
RUN_EVAL = True
# If None, auto-pick lowest SASV-EER calibrated method on dev (excludes raw_sum)
FORCE_METHOD = None  # e.g. "logit_sum_sasv"

## 1. Load **dev** scores and fit calibrators

In [3]:
s_asv_dev, s_cm_dev, keys_dev = load_score_csv(DEV_CSV)
print(f"dev trials: {len(keys_dev)}")

calibrators = fit_calibrators_dev(s_asv_dev, s_cm_dev, keys_dev)
ckpt = OUT_DIR / "calibrators_dev.pkl"
save_calibrators(calibrators, ckpt)
print("Saved", ckpt)

dev trials: 29548
Saved D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019\runs\ecapa_plus_lfcc_calibrated\calibrators_dev.pkl


## 2. Compare fusion methods on **dev**

In [4]:
dev_results = evaluate_methods(
    s_asv_dev,
    s_cm_dev,
    keys_dev,
    calibrators,
    sasv_root=DEFAULT_SASV,
)

print("Dev EERs:")
for name, m in sorted(dev_results.items(), key=lambda kv: kv[1]["sasv_eer"]):
    print(
        f"  {name:22s}  SASV={m['sasv_eer_percent']:7.4f}%  "
        f"SV={m['sv_eer_percent']:7.4f}%  SPF={m['spf_eer_percent']:7.4f}%"
    )

LOCKED_METHOD = FORCE_METHOD or pick_best_method(dev_results)
print("\nLocked method:", LOCKED_METHOD)
print("Dev locked:", {k: dev_results[LOCKED_METHOD][k] for k in [
    "sasv_eer_percent", "sv_eer_percent", "spf_eer_percent"
]})

Dev EERs:
  platt_sum_sv_spf        SASV= 0.8086%  SV= 1.3350%  SPF= 0.1032%
  joint_proba             SASV= 0.8086%  SV= 1.7520%  SPF= 0.2022%
  joint_logit             SASV= 0.8086%  SV= 1.7520%  SPF= 0.2022%
  logit_sum_sasv          SASV= 0.8086%  SV= 1.7520%  SPF= 0.2022%
  logit_sum_sv_spf        SASV= 1.0108%  SV= 1.9542%  SPF= 0.1348%
  raw_sum                 SASV= 1.1438%  SV= 2.0978%  SPF= 0.0897%
  platt_sum_sasv          SASV= 1.4151%  SV= 1.7520%  SPF= 1.3477%

Locked method: platt_sum_sv_spf
Dev locked: {'sasv_eer_percent': 0.808625336908929, 'sv_eer_percent': 1.3349514562152576, 'spf_eer_percent': 0.1031575170142425}


In [5]:
save_run_summary(
    split="dev",
    locked_method=LOCKED_METHOD,
    results=dev_results,
    output_dir=OUT_DIR / "dev",
    extra={"source_csv": str(DEV_CSV), "calibrators": str(ckpt)},
)
(OUT_DIR / "locked_method.json").write_text(
    json.dumps(
        {
            "locked_method": LOCKED_METHOD,
            "tuned_on": "dev",
            "objective": "min SASV-EER among calibrated methods",
            "dev_metrics": dev_results[LOCKED_METHOD],
        },
        indent=2,
    ),
    encoding="utf-8",
)
print("Wrote", OUT_DIR / "dev")

Wrote D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019\runs\ecapa_plus_lfcc_calibrated\dev


## 3. Locked **eval** (same calibrators + method — do not re-fit)

In [6]:
if not RUN_EVAL:
    print("RUN_EVAL=False — skip eval")
else:
    s_asv_ev, s_cm_ev, keys_ev = load_score_csv(EVAL_CSV)
    print(f"eval trials: {len(keys_ev)} | method={LOCKED_METHOD}")

    eval_results = evaluate_methods(
        s_asv_ev,
        s_cm_ev,
        keys_ev,
        calibrators,  # fitted on dev only
        sasv_root=DEFAULT_SASV,
    )

    print("Eval EERs (calibrators frozen from dev):")
    for name, m in sorted(eval_results.items(), key=lambda kv: kv[1]["sasv_eer"]):
        mark = " <-- locked" if name == LOCKED_METHOD else ""
        print(
            f"  {name:22s}  SASV={m['sasv_eer_percent']:7.4f}%  "
            f"SV={m['sv_eer_percent']:7.4f}%  SPF={m['spf_eer_percent']:7.4f}%{mark}"
        )

    save_run_summary(
        split="eval",
        locked_method=LOCKED_METHOD,
        results=eval_results,
        output_dir=OUT_DIR / "eval",
        extra={
            "source_csv": str(EVAL_CSV),
            "calibrators": str(ckpt),
            "note": "Calibrators fitted on dev only; method locked on dev",
        },
    )

    locked = {
        "locked_method": LOCKED_METHOD,
        "tuned_on": "dev",
        "dev_metrics": dev_results[LOCKED_METHOD],
        "eval_metrics": eval_results[LOCKED_METHOD],
        "eval_raw_sum": eval_results["raw_sum"],
    }
    (OUT_DIR / "locked_alpha.json").write_text(
        json.dumps(locked, indent=2), encoding="utf-8"
    )
    # clearer name
    (OUT_DIR / "locked_eval.json").write_text(
        json.dumps(locked, indent=2), encoding="utf-8"
    )
    print("\nReport (locked):")
    print(json.dumps(locked, indent=2))

eval trials: 102579 | method=platt_sum_sv_spf
Eval EERs (calibrators frozen from dev):
  raw_sum                 SASV= 7.1279%  SV= 1.5642%  SPF= 9.7070%
  logit_sum_sv_spf        SASV= 7.3557%  SV= 1.3966%  SPF= 9.8838%
  logit_sum_sasv          SASV= 7.5003%  SV= 1.2452%  SPF=10.0560%
  joint_proba             SASV= 7.5086%  SV= 1.2392%  SPF=10.0576%
  joint_logit             SASV= 7.5086%  SV= 1.2392%  SPF=10.0576%
  platt_sum_sasv          SASV= 7.5419%  SV= 1.2452%  SPF=10.0931%
  platt_sum_sv_spf        SASV= 8.1423%  SV= 1.0987%  SPF=10.9827% <-- locked

Report (locked):
{
  "locked_method": "platt_sum_sv_spf",
  "tuned_on": "dev",
  "dev_metrics": {
    "sasv_eer": 0.00808625336908929,
    "sv_eer": 0.013349514562152576,
    "spf_eer": 0.001031575170142425,
    "sasv_eer_percent": 0.808625336908929,
    "sv_eer_percent": 1.3349514562152576,
    "spf_eer_percent": 0.1031575170142425,
    "method": "platt_sum_sv_spf"
  },
  "eval_metrics": {
    "sasv_eer": 0.08142250203198909,
 

## Done

Report **eval** EERs for the **dev-locked** method. Do not pick the method from eval.

Outputs under `runs/ecapa_plus_lfcc_calibrated/`:

- `calibrators_dev.pkl`
- `locked_method.json` / `locked_eval.json`
- `dev/metrics_dev.json`, `eval/metrics_eval.json`